# Indexar la biblioteca de PDF → índice para el portal

Ejecuta las celdas **en orden**. Todo corre en la nube de Google (gratis); no descarga nada a tu Mac.

Necesitas tu **API key de Gemini**: https://aistudio.google.com/apikey


In [ ]:
# 1) Autenticación con tu cuenta Google (para leer la carpeta de PDF de Drive)
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
drive = build('drive', 'v3')
print('OK autenticado')

In [ ]:
# 2) Localizar los PDF dentro de la carpeta (por ID, incluye subcarpetas)
FOLDER_ID = "1cuQ8dpDkQg7lSFOBeP6Lf30gRP-7jHvv"

def list_pdfs(folder_id):
    pdfs, stack = [], [folder_id]
    while stack:
        fid = stack.pop()
        token = None
        while True:
            resp = drive.files().list(
                q=f"'{fid}' in parents and trashed=false",
                fields="nextPageToken, files(id,name,mimeType)",
                pageToken=token, pageSize=1000,
                supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
            for f in resp.get('files', []):
                if f['mimeType'] == 'application/vnd.google-apps.folder':
                    stack.append(f['id'])
                elif f['mimeType'] == 'application/pdf' or f['name'].lower().endswith('.pdf'):
                    pdfs.append(f)
            token = resp.get('nextPageToken')
            if not token:
                break
    return pdfs

pdfs = list_pdfs(FOLDER_ID)
print(len(pdfs), 'PDF encontrados')

In [ ]:
# 3) Descargar los PDF a la máquina temporal de Colab
import os
from googleapiclient.http import MediaIoBaseDownload

CORPUS = '/content/corpus'
os.makedirs(CORPUS, exist_ok=True)
seen = {}
for f in pdfs:
    name = f['name']
    if name in seen:                      # evita colisiones de nombre
        name = f"{f['id'][:6]}_{name}"
    seen[name] = True
    dest = os.path.join(CORPUS, name)
    if os.path.exists(dest):
        continue
    req = drive.files().get_media(fileId=f['id'])
    with open(dest, 'wb') as fh:
        dl = MediaIoBaseDownload(fh, req)
        done = False
        while not done:
            _, done = dl.next_chunk()
print('Descargados', len(os.listdir(CORPUS)), 'archivos en', CORPUS)

In [ ]:
# 4) Traer el código (rag/) desde GitHub e instalar dependencias
import os
if not os.path.exists('/content/Generador-SSC'):
    !git clone -q https://github.com/FranR78/Generador-SSC.git /content/Generador-SSC
else:
    !cd /content/Generador-SSC && git pull -q
!pip -q install -r /content/Generador-SSC/rag/requirements.txt
print('OK código + dependencias')

In [ ]:
# 5) Tu API key y dónde guardar el índice (en tu Drive, para que persista)
import os, getpass
from google.colab import drive as gdrive
gdrive.mount('/content/drive')

os.environ['GEMINI_API_KEY'] = getpass.getpass('Pega tu API key de Gemini: ')
os.environ['SSC_CORPUS_DIR'] = '/content/corpus'
os.environ['SSC_INDEX_DIR'] = '/content/drive/MyDrive/Generador-SSC-index'
print('El índice se guardará en:', os.environ['SSC_INDEX_DIR'])

In [ ]:
# 6) Trocear los PDF (rápido, sin coste)
!cd /content/Generador-SSC/rag && python ingest.py "$SSC_CORPUS_DIR" 

In [ ]:
# 7) Crear el índice de embeddings (usa tu key; es resumible si se corta)
!cd /content/Generador-SSC/rag && python embed.py

In [ ]:
# 8) Prueba: preguntar a la biblioteca
!cd /content/Generador-SSC/rag && python query.py "¿Qué presión de trabajo tiene el circuito de alta?"